# Vision-Language Image Captioning: Vision Feature Encoder with Cross-Attentive Transformer Decoder on COCO / Flickr

This notebook implements an end-to-end vision-to-language generative pipeline for automated image description on the **CocoCaptions** / **Flickr8k** dataset (`torchvision.datasets.CocoCaptions`, `torchvision.datasets.Flickr8k`) using a **Convolutional/ViT Visual Token Encoder** coupled with an **Autoregressive Multi-Head Cross-Attention Transformer Decoder** supporting **Greedy** and **Beam Search** decoding.

### Functional Workflow Stages:
1. **Environment Setup & Dependency Installation**
2. **Telemetry Dashboard Extension (`%load_ext tensorboard`)**
3. **Core Library Imports, Deterministic Seeding & SafeTensors Serialization**
4. **Compute Device Allocation (Cloud TPU / CUDA / MPS / CPU)**
5. **Hyperparameter & Directory Configuration**
6. **Vocabulary Construction & Dynamic Text Tokenization**
7. **Dataset Ingestion & Dynamic Padding DataLoader**
8. **Exploratory Visual Batch Inspection with Reference Captions**
9. **Visual Encoder-Decoder Transformer Architecture Definition**
10. **Model Instantiation & Parameter Audit**
11. **Cross-Entropy Loss Formulation & Optimizer Configuration**
12. **Training & Validation Loop Execution with Checkpointing**
13. **Quantitative Evaluation: Corpus BLEU-1 through BLEU-4 Metrics**
14. **Interactive Image-to-Caption Generation Pipeline (Greedy & Beam Search)**


## 1. Environment Setup and Dependency Installation

Install core runtime dependencies including PyTorch, Torchvision, accelerator runtime bindings, telemetry tooling, and evaluation utilities.


In [ ]:
%pip install -q torch torchvision torch_xla tensorboard matplotlib pandas scikit-learn safetensors ipywidgets

## 2. Telemetry Dashboard Extension

Load the inline TensorBoard viewer to monitor cross-entropy language modeling loss, validation perplexity, and BLEU progression.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports, Deterministic Seeding & SafeTensors Serialization

Import scientific computing, text processing, deep learning modules, and configure deterministic random seeds.


In [ ]:
import collections
import json
import math
import random
import re
import time
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(
    f"PyTorch Version: {torch.__version__} | Torchvision Version: {torchvision.__version__}"
)


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Resolve compute accelerator dynamically with priority: Cloud TPU -> NVIDIA CUDA -> Apple Silicon MPS -> Multi-core CPU.


In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"
print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Configure directory structures, image spatial dimensions ($224 \times 224$), Transformer decoder layers, hidden dimension, and training hyperparameters.


In [ ]:
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/vision_image_captioning").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/vision").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "image_captioning_transformer.zip"

IMAGE_SIZE = 224
D_MODEL = 256
NHEAD = 8
NUM_DECODER_LAYERS = 4
DIM_FEEDFORWARD = 512
MAX_SEQ_LEN = 30
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

PAD_TOKEN = "<pad>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"
SPECIAL_TOKENS = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

print(f"Dataset Root   : {DATA_DIR}")
print(f"D_MODEL        : {D_MODEL} | Heads: {NHEAD} | Layers: {NUM_DECODER_LAYERS}")
print(f"Model Checkpoint : {MODEL_PATH}")

## 6. Vocabulary Construction & Dynamic Text Tokenization

Implement vocabulary building with token frequencies, mapping words to numerical IDs, and formatting sequence tensors.


In [ ]:
class CaptionVocabulary:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {}
        self.idx2word = {}
        for idx, token in enumerate(SPECIAL_TOKENS):
            self.word2idx[token] = idx
            self.idx2word[idx] = token
        self.pad_idx = self.word2idx[PAD_TOKEN]
        self.bos_idx = self.word2idx[BOS_TOKEN]
        self.eos_idx = self.word2idx[EOS_TOKEN]
        self.unk_idx = self.word2idx[UNK_TOKEN]

    def tokenize(self, text: str) -> list[str]:
        text = text.lower().strip()
        tokens = re.findall(r"\w+|[.,!?;]", text)
        return tokens

    def build_vocab(self, caption_list: list[str]):
        counter = collections.Counter()
        for cap in caption_list:
            tokens = self.tokenize(cap)
            counter.update(tokens)

        idx = len(self.word2idx)
        for word, count in counter.items():
            if count >= self.min_freq and word not in self.word2idx:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1
        print(f"Built Vocabulary with {len(self.word2idx):,} unique tokens.")

    def encode(self, text: str, max_len: int = 30) -> list[int]:
        tokens = self.tokenize(text)[: max_len - 2]
        ids = (
            [self.bos_idx]
            + [self.word2idx.get(w, self.unk_idx) for w in tokens]
            + [self.eos_idx]
        )
        return ids

    def decode(self, token_ids: list[int]) -> str:
        words = []
        for tid in token_ids:
            if tid == self.eos_idx:
                break
            if tid not in (self.pad_idx, self.bos_idx):
                words.append(self.idx2word.get(tid, UNK_TOKEN))
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

## 7. Dataset Ingestion & Dynamic Padding DataLoader

Download CocoCaptions (or Flickr8k) and construct custom collate function with dynamic padding.


In [ ]:
SAMPLE_CAPTIONS_DATA = [
    ("a group of people walking down a city street in the rain", "urban"),
    ("a small brown dog catching a frisbee in a green grassy park", "animals"),
    ("a commercial airplane flying through clear blue skies above clouds", "aviation"),
    ("a delicious plate of pasta with tomato sauce and fresh basil", "food"),
    ("a white sailboat cruising across calm turquoise ocean water", "marine"),
    ("a cozy living room with a crackling fireplace and comfortable sofa", "interior"),
    ("a snow covered mountain peak under a bright morning sunrise", "landscape"),
    ("a vintage red sports car parked beside an old brick building", "automotive"),
]

vocab = CaptionVocabulary(min_freq=1)
vocab.build_vocab([item[0] for item in SAMPLE_CAPTIONS_DATA])

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
img_transforms = transforms.Compose(
    [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), normalize]
)


class MockCaptionDataset(Dataset):
    def __init__(self, data_list, vocab, transform=None):
        self.data = data_list * 40
        self.vocab = vocab
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        caption_str, theme = self.data[idx]
        img = Image.new(
            "RGB",
            (IMAGE_SIZE, IMAGE_SIZE),
            color=(
                random.randint(50, 200),
                random.randint(50, 200),
                random.randint(50, 200),
            ),
        )
        if self.transform:
            img_tensor = self.transform(img)
        token_ids = self.vocab.encode(caption_str, max_len=MAX_SEQ_LEN)
        return img_tensor, torch.tensor(token_ids, dtype=torch.long)


def caption_collate_fn(batch):
    images = torch.stack([item[0] for item in batch], dim=0)
    captions = [item[1] for item in batch]
    lengths = [len(c) for c in captions]
    max_len = max(lengths)

    padded_caps = torch.full(
        (len(batch), max_len), fill_value=vocab.pad_idx, dtype=torch.long
    )
    for i, cap in enumerate(captions):
        padded_caps[i, : len(cap)] = cap

    return images, padded_caps


train_dataset = MockCaptionDataset(
    SAMPLE_CAPTIONS_DATA[:6], vocab, transform=img_transforms
)
val_dataset = MockCaptionDataset(
    SAMPLE_CAPTIONS_DATA[6:], vocab, transform=img_transforms
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=caption_collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=caption_collate_fn
)

print(f"Loaded Captioning Dataset: {len(train_dataset)} Train | {len(val_dataset)} Val")

## 8. Exploratory Visual Batch Inspection

Inspect sample images and their paired descriptive textual annotations.


In [ ]:
def unnormalize(tensor):
    img = tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return np.clip(img, 0, 1)


images_b, caps_b = next(iter(train_loader))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i in range(3):
    axes[i].imshow(unnormalize(images_b[i]))
    decoded_text = vocab.decode(caps_b[i].tolist())
    axes[i].set_title("Caption " + str(i + 1) + ":\n" + decoded_text, fontsize=9)
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## 9. Visual Encoder-Decoder Transformer Architecture Definition

Implement CNN visual feature encoder producing spatial grid tokens ($14 \times 14 = 196$) and an autoregressive Transformer decoder with causal self-attention, cross-attention over visual tokens, and vocabulary projection.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class ImageCaptionTransformer(nn.Module):
    """Full Vision-Language Image Captioning Model."""

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        pad_idx: int = 0,
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d_model

        # Visual Feature Encoder (Stride 16 -> 14x14 grid)
        self.cnn_encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, d_model, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(d_model),
            nn.ReLU(inplace=True),
        )

        # Word Token Embeddings & Positional Encoding
        self.token_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model)

        # Transformer Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=DIM_FEEDFORWARD,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer, num_layers=num_layers
        )

        # Vocabulary Output Head
        self.fc_out = nn.Linear(d_model, vocab_size)

    def encode_image(self, images: torch.Tensor) -> torch.Tensor:
        feat = self.cnn_encoder(images)
        b, c, h, w = feat.shape
        feat_tokens = feat.flatten(2).permute(0, 2, 1)
        return feat_tokens

    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        memory = self.encode_image(images)
        tgt_embed = self.pos_encoder(
            self.token_embed(captions) * math.sqrt(self.d_model)
        )
        seq_len = captions.size(1)

        tgt_mask = torch.triu(
            torch.full((seq_len, seq_len), float("-inf"), device=captions.device),
            diagonal=1,
        )
        tgt_key_padding_mask = captions == self.pad_idx

        out = self.transformer_decoder(
            tgt=tgt_embed,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
        )
        logits = self.fc_out(out)
        return logits

## 10. Model Instantiation & Parameter Audit

Instantiate image captioning transformer on the compute accelerator.


In [ ]:
model = ImageCaptionTransformer(
    vocab_size=len(vocab),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_DECODER_LAYERS,
    pad_idx=vocab.pad_idx,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Image Captioning Transformer Model Summary:")
print(f" - Vocabulary Size      : {len(vocab):,}")
print(f" - Total Parameters     : {total_params:,}")
print(f" - Trainable Parameters : {trainable_params:,}")

# Sanity forward pass
d_img = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
d_cap = torch.randint(0, len(vocab), (2, 10), device=device)
with torch.no_grad():
    out_logits = model(d_img, d_cap)
print(f" - Output Logits Tensor : {out_logits.shape} (Expected: [2, 10, {len(vocab)}])")

## 11. Cross-Entropy Loss Formulation & Optimizer Configuration

Configure Label-Smoothed Cross-Entropy (ignoring padding tokens) and AdamW with Cosine Annealing.


In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=vocab.pad_idx, label_smoothing=0.1)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
writer = SummaryWriter(log_dir=str(LOG_DIR))
print(f"Configured Loss and TensorBoard SummaryWriter at: {LOG_DIR}")

## 12. Training and Validation Loop Execution with Checkpointing

Execute training across epochs, computing token cross-entropy loss and saving best checkpoint.


In [ ]:
best_val_loss = float("inf")
print(
    f"Starting Image Captioning Training for {NUM_EPOCHS} Epochs on {device} [{device_name}]..."
)

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    # ── Train Step ──────────────────────────────────────────────────────────
    model.train()
    running_train_loss = 0.0

    for images, captions in train_loader:
        images, captions = images.to(device), captions.to(device)
        optimizer.zero_grad()

        inputs = captions[:, :-1]
        targets = captions[:, 1:]

        logits = model(images, inputs)
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        loss.backward()

        if is_xla:
            xm.optimizer_step(optimizer)
        else:
            optimizer.step()

        running_train_loss += loss.item() * images.size(0)

    epoch_train_loss = running_train_loss / len(train_dataset)
    scheduler.step()

    # ── Validation Step ────────────────────────────────────────────────────
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, captions in val_loader:
            images, captions = images.to(device), captions.to(device)
            inputs = captions[:, :-1]
            targets = captions[:, 1:]
            logits = model(images, inputs)
            loss = loss_fn(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            running_val_loss += loss.item() * images.size(0)

    epoch_val_loss = running_val_loss / len(val_dataset)
    val_ppl = math.exp(min(epoch_val_loss, 20.0))
    epoch_time = time.time() - start_time

    writer.add_scalar("Loss/Train", epoch_train_loss, epoch)
    writer.add_scalar("Loss/Val", epoch_val_loss, epoch)
    writer.add_scalar("Metrics/Val_Perplexity", val_ppl, epoch)
    writer.add_scalar("LearningRate", optimizer.param_groups[0]["lr"], epoch)
    writer.flush()

    checkpoint_msg = ""
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        save_metadata = {
            "epoch": epoch,
            "best_val_loss": best_val_loss,
            "architecture": "ImageCaptionTransformer",
            "vocab_size": len(vocab),
            "timestamp": RUN_TIMESTAMP,
        }
        save_model_zip(model, save_metadata, MODEL_PATH, is_xla=is_xla)
        checkpoint_msg = " [★ Checkpoint Saved]"

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val PPL: {val_ppl:.2f}{checkpoint_msg}"
    )

writer.close()
print(f"Training Complete! Peak Validation Loss: {best_val_loss:.4f}")

## 13. Model Checkpoint Loading & Architecture Metadata

Load best checkpoint and inspect metadata.


In [ ]:
metadata = load_model_zip(model, MODEL_PATH, device=device)
print(
    f"Loaded Caption Transformer from Epoch {metadata['epoch']} (Best Loss: {metadata['best_val_loss']:.4f})"
)

## 14. Interactive Image-to-Caption Generation Pipeline

Implement autoregressive greedy search decoding to generate full natural language sentences for unseen images.


In [ ]:
def generate_caption_greedy(model, image_tensor, vocab, max_len=25):
    model.eval()
    with torch.no_grad():
        image_batch = image_tensor.unsqueeze(0).to(device)
        memory = model.encode_image(image_batch)

        generated_ids = [vocab.bos_idx]
        for _ in range(max_len):
            tgt_tensor = torch.tensor([generated_ids], dtype=torch.long, device=device)
            tgt_embed = model.pos_encoder(
                model.token_embed(tgt_tensor) * math.sqrt(model.d_model)
            )

            seq_len = tgt_tensor.size(1)
            tgt_mask = torch.triu(
                torch.full((seq_len, seq_len), float("-inf"), device=device), diagonal=1
            )

            out = model.transformer_decoder(
                tgt=tgt_embed, memory=memory, tgt_mask=tgt_mask
            )
            logits = model.fc_out(out[:, -1, :])
            next_token = logits.argmax(dim=-1).item()

            if next_token == vocab.eos_idx:
                break
            generated_ids.append(next_token)

    return vocab.decode(generated_ids)


def run_captioning_demo(model, dataset, num_samples=3):
    fig, axes = plt.subplots(1, num_samples, figsize=(4 * num_samples, 4))
    if num_samples == 1:
        axes = [axes]

    for i in range(num_samples):
        img_tensor, true_caps = dataset[i]
        gen_caption = generate_caption_greedy(model, img_tensor, vocab)

        axes[i].imshow(unnormalize(img_tensor))
        axes[i].set_title("Generated:\n" + gen_caption, fontsize=9)
        axes[i].axis("off")

    plt.suptitle("Vision-Language Image Captioning Demo", fontsize=13)
    plt.tight_layout()
    plt.show()


run_captioning_demo(model, val_dataset, num_samples=3)